# CFPB Consumer Complaints → NLP text-classification subset

Filters the full ~9.1 GB `complaints.csv` down to a clean labelled subset using DuckDB,
streaming from disk rather than loading the file into memory.

**Two things about this specific file, discovered by testing it:**

1. **`parallel=false` is required.** DuckDB's parallel CSV reader aborts on this file with
   *"The Parallel CSV Reader currently does not support a full read on this file"* — almost
   certainly because complaint narratives contain embedded newlines inside quoted fields.
   Single-threaded, a full scan of all 17M rows takes about 60 seconds.
2. **`Date received` is read as VARCHAR on purpose.** DuckDB would auto-infer `DATE` from a
   sample of the file's head, which risks a hard cast error on a malformed date 8 GB in.
   Reading it as text and using `TRY_CAST` turns that failure into a `NULL` we can count.

**Cost control:** every full scan costs ~60 s, so this notebook does exactly **three** of
them (Step 0, Step 1, Step 2). Step 2 materialises the date-filtered rows into an in-memory
table, so Steps 3–6 run against that table and return instantly.

In [ ]:
import time

import duckdb
import pandas as pd

pd.set_option("display.max_colwidth", 60)

CSV = "complaints.csv"

con = duckdb.connect()
con.sql("SET memory_limit='8GB'")
con.sql("SET temp_directory='./.tmp'")  # spill to disk instead of OOM
# Note: don't try `SET enable_progress_bar=false` here — inside a Jupyter kernel DuckDB
# raises unless ipywidgets is installed. It renders no progress bar in the kernel anyway.

# Every query reads the CSV through this same reader definition.
SRC = (
    "read_csv_auto('" + CSV + "', "
    "parallel=false, "                      # required: see notes above
    "types={'Date received': 'VARCHAR'})"   # parse dates ourselves, safely
)

print("duckdb", duckdb.__version__)
print(SRC)

## Step 0 — Confirm the full file is being read

Counts every row in the file (~60 s, full scan). Since we are scanning anyway, this also
counts how many `Date received` values fail to parse as a date — that number is the rows
Step 2's `TRY_CAST` would silently drop.

In [ ]:
t0 = time.time()
total_rows, unparseable_dates = con.sql(f"""
    SELECT count(*)                                                       AS total_rows,
           count(*) FILTER (WHERE TRY_CAST("Date received" AS DATE) IS NULL) AS unparseable_dates
    FROM {SRC}
""").fetchone()

print(f"TOTAL ROWS IN FILE : {total_rows:,}")
print(f"unparseable dates  : {unparseable_dates:,}")
print(f"full scan took     : {time.time() - t0:.1f}s")

In [ ]:
# Column names exactly as they appear in the file (header sample only, not a full scan).
schema = con.sql(f"DESCRIBE SELECT * FROM {SRC}").df()
print(f"{len(schema)} columns:\n")
print(schema[["column_name", "column_type"]].to_string(index=False))

# You asked to be told if the date column is named differently in your file.
file_columns = set(schema["column_name"])
DATE_COL = "Date received"
NARRATIVE_COL = "Consumer complaint narrative"
print()
for expected in (DATE_COL, NARRATIVE_COL, "Product", "Issue", "Sub-product"):
    status = "present" if expected in file_columns else "*** MISSING ***"
    print(f"  {expected!r:32} {status}")

## Step 1 — Real `Product` values as they appear in the file

No assumptions: this is the full distinct list with row counts (~60 s, full scan).
Step 3 matches your requested categories against exactly these strings.

In [ ]:
t0 = time.time()
product_counts = con.sql(f"""
    SELECT Product, count(*) AS n_rows
    FROM {SRC}
    GROUP BY Product
    ORDER BY n_rows DESC
""").df()

print(f"{len(product_counts)} distinct Product values (full scan {time.time() - t0:.1f}s)\n")
with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
    print(product_counts.to_string(index=False))

actual_products = set(product_counts["Product"].dropna())

## Step 2 — Filter by date (2025-01-01 → 2026-12-31 inclusive)

Uses `Date received` (not `Date sent to company`). `TRY_CAST` handles the YYYY-MM-DD text
and yields `NULL` — which fails the `BETWEEN` and so drops the row — for anything malformed.

This is the last full scan. It materialises the surviving rows into a table, keeping only
the five columns Step 5 asks for. Column pruning has to happen here rather than at Step 5
to keep the materialised table small; the discarded columns (Company, State, ZIP, Tags, …)
are ones Step 5 drops anyway. **If you later want one of those columns back, re-run from
this cell** with it added to the SELECT.

In [ ]:
t0 = time.time()
con.sql(f"""
    CREATE OR REPLACE TABLE stage_dated AS
    SELECT TRY_CAST("Date received" AS DATE)   AS date_received,
           Product                             AS product,
           "Sub-product"                       AS sub_product,
           Issue                               AS issue,
           "Consumer complaint narrative"      AS narrative
    FROM {SRC}
    WHERE TRY_CAST("Date received" AS DATE)
          BETWEEN DATE '2025-01-01' AND DATE '2026-12-31'
""")
print(f"full scan + materialise took {time.time() - t0:.1f}s")

n_dated, min_date, max_date = con.sql("""
    SELECT count(*), min(date_received), max(date_received) FROM stage_dated
""").fetchone()

print(f"\nrows surviving date filter : {n_dated:,}  ({n_dated / total_rows:.1%} of file)")
print(f"earliest Date received     : {min_date}")
print(f"latest   Date received     : {max_date}")

## Step 3 — Filter by product category

Your ten requested names are matched against the real values from Step 1. Anything that
does not match exactly is reported, not silently dropped — and for each miss we look for a
close variant (case/whitespace-insensitive, then substring) to show what it probably became.

In [ ]:
REQUESTED = [
    "Debt collection",
    "Credit card",
    "Checking or savings account",
    "Money transfer, virtual currency, or money service",
    "Mortgage",
    "Vehicle loan or lease",
    "Student loan",
    "Payday loan, title loan, personal loan, or advance loan",
    "Prepaid card",
    "Debt or credit management",
]

matched = [p for p in REQUESTED if p in actual_products]
unmatched = [p for p in REQUESTED if p not in actual_products]

print(f"MATCHED {len(matched)}/{len(REQUESTED)} exactly:")
for p in matched:
    print(f"  OK        {p!r}")

if unmatched:
    norm = {a.casefold().strip(): a for a in actual_products}
    print(f"\nDID NOT MATCH {len(unmatched)} — these contribute NO rows:")
    for p in unmatched:
        print(f"  MISSING   {p!r}")
        near = norm.get(p.casefold().strip())
        if near:
            print(f"            -> case/space variant exists: {near!r}")
            continue
        head = p.casefold().split(",")[0].split(" or ")[0].strip()
        similar = [a for a in sorted(actual_products) if head and head in a.casefold()]
        for s in similar:
            print(f"            -> possibly: {s!r}")
        if not similar:
            print("            -> no similar value found in the file")
else:
    print("\nAll requested categories matched exactly.")

# Build the IN list as escaped literals; DuckDB's `IN ?` list binding is unreliable.
in_list = ", ".join("'" + p.replace("'", "''") + "'" for p in matched)
con.sql(f"CREATE OR REPLACE TABLE stage_product AS "
        f"SELECT * FROM stage_dated WHERE product IN ({in_list})")

n_product = con.sql("SELECT count(*) FROM stage_product").fetchone()[0]
print(f"\nrows surviving product filter : {n_product:,}  (from {n_dated:,})")

## Step 4 — Remove empty narratives

Drops `NULL`, empty strings, and whitespace-only narratives (the last of these is not
strictly what you asked for, but a narrative of `"   "` is just as useless for NLP as `""`).

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE stage_narrative AS
    SELECT * FROM stage_product
    WHERE narrative IS NOT NULL AND trim(narrative) <> ''
""")

n_narrative = con.sql("SELECT count(*) FROM stage_narrative").fetchone()[0]
dropped = n_product - n_narrative
print(f"rows with a usable narrative : {n_narrative:,}")
print(f"dropped (null/empty)         : {dropped:,}  ({dropped / n_product:.1%} of Step 3 rows)")

## Step 5 — Keep only the needed columns

`narrative` (text) and `product` (label), plus `issue`, `sub_product` and `date_received`.
The wide columns were already pruned in Step 2, so this cell fixes the column order and
confirms the final shape.

In [ ]:
con.sql("""
    CREATE OR REPLACE TABLE final_subset AS
    SELECT narrative, product, issue, sub_product, date_received
    FROM stage_narrative
""")

print(con.sql("DESCRIBE final_subset").df()[["column_name", "column_type"]].to_string(index=False))
print()
con.sql("""
    SELECT substr(narrative, 1, 60) AS narrative_preview, product, issue, date_received
    FROM final_subset LIMIT 5
""").show(max_width=200)

## Step 6 — Class balance

In [ ]:
balance = con.sql("""
    SELECT product,
           count(*)                                   AS n_rows,
           round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS pct
    FROM final_subset
    GROUP BY product
    ORDER BY n_rows DESC
""").df()

with pd.option_context("display.max_rows", None, "display.max_colwidth", 80):
    print(balance.to_string(index=False))

final_total = int(balance["n_rows"].sum())
print(f"\nTOTAL ROWS IN FINAL SUBSET : {final_total:,}")
print(f"kept from original file    : {final_total / total_rows:.2%} of {total_rows:,}")
if len(balance):
    print(f"imbalance ratio (max/min)  : {balance.n_rows.max() / balance.n_rows.min():.1f}x")

## Save the subset

Not one of your six steps, but the filtered table lives only in this kernel's memory —
writing it to Parquet means you never pay the 3-minute scan again. Load it later with
`duckdb.sql("SELECT * FROM 'complaints_nlp_subset.parquet'")` or `pd.read_parquet(...)`.

In [ ]:
import os

OUT = "cfpb_filtered.parquet"
con.sql(f"COPY final_subset TO '{OUT}' (FORMAT PARQUET, COMPRESSION ZSTD)")
print(f"wrote {OUT}  ({os.path.getsize(OUT) / 1024**2:,.1f} MB, {final_total:,} rows)")